# FIM軌跡 インタラクティブダッシュボード

## 概要
FIM（Functional Independence Measure）の縦断軌跡データを読み込み、潜在クラス分析を行うインタラクティブダッシュボードです。

### 機能
1. **データ読み込み** - CSVファイルからデータをロード
2. **全員FIM軌跡表示** - 全患者のFIM軌跡を可視化
3. **インタラクティブダッシュボード**:
   - 0週FIMスコアがない症例の除外チェックボックス
   - 0週FIMスコアの下限・上限フィルタ
   - Week since admission の範囲（上限）フィルタ
   - 情報量基準（BIC）による最適クラスタ数の自動決定
   - Individual Trajectories (sampled) + Class Means 描画
   - クラスタ別 Table 1 描画

### 高速化
- R/rpy2の代わりにscikit-learnのGaussianMixtureを使用（数分→数十ミリ秒）
- 患者ごとの軌跡特徴量（切片・傾き）を事前抽出し、GMM でクラスタリング
- BIC による最適クラスタ数選択（sklearn内蔵）
- debounce付きウィジェットで過度な再計算を防止

### 必要パッケージ
```
pip install numpy pandas matplotlib scikit-learn scipy ipywidgets tableone ipympl jupyterlab
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from tableone import TableOne
import warnings
import time
import os

warnings.filterwarnings('ignore')

# --- matplotlib backend ---
# ipympl (%matplotlib widget) が使えればインタラクティブ、
# なければ inline にフォールバック
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    print('Backend: ipympl (interactive)')
except Exception:
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
        print('Backend: inline (static)')
    except Exception:
        pass

print('Setup complete.')


## 1. データ読み込み

下のセルの `DATA_PATH` を自分のCSVファイルのパスに変更してください。  
または `FileUpload` ウィジェットでアップロードすることもできます。


In [ ]:
# ====================================================
# データ読み込み
# ====================================================
# 方法1: ファイルパスを直接指定（こちらを優先）
DATA_PATH = ""  # 例: r"C:\\Users\\user\\Downloads\\moriyama_lcmm_rwd.csv"

# 方法2: ファイルアップロードウィジェット
uploader = widgets.FileUpload(accept='.csv', multiple=False, description='CSVアップロード')
load_btn = widgets.Button(description='ファイルパスから読み込み', button_style='primary')
status_label = widgets.HTML(value='<i>CSVファイルをアップロードするか、DATA_PATHを設定してload_btnを押してください</i>')

raw_data = None  # グローバル変数

def _load_from_path(path):
    global raw_data
    raw_data = pd.read_csv(path)
    status_label.value = f'<b style="color:green">読み込み完了: {len(raw_data)} 行, {raw_data.shape[1]} 列</b>'
    print(f"Columns: {list(raw_data.columns)}")
    print(f"Shape: {raw_data.shape}")
    display(raw_data.head())

def on_upload_change(change):
    global raw_data
    if uploader.value:
        import io
        uploaded = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        content = uploaded['content'] if isinstance(uploaded, dict) else uploaded.content
        raw_data = pd.read_csv(io.BytesIO(content))
        status_label.value = f'<b style="color:green">アップロード読み込み完了: {len(raw_data)} 行, {raw_data.shape[1]} 列</b>'
        print(f"Columns: {list(raw_data.columns)}")
        print(f"Shape: {raw_data.shape}")
        display(raw_data.head())

def on_load_btn(b):
    if DATA_PATH and os.path.exists(DATA_PATH):
        _load_from_path(DATA_PATH)
    else:
        status_label.value = '<b style="color:red">DATA_PATHが未設定または存在しません</b>'

uploader.observe(on_upload_change, names='value')
load_btn.on_click(on_load_btn)

display(widgets.VBox([
    widgets.HBox([uploader, load_btn]),
    status_label
]))

# DATA_PATHが既に設定されていれば自動読み込み
if DATA_PATH and os.path.exists(DATA_PATH):
    _load_from_path(DATA_PATH)


## 2. データ前処理

列名のリネーム、重複除外、3回以上観測のある患者のみ抽出を行います。


In [ ]:
# ====================================================
# データ前処理
# ====================================================
assert raw_data is not None, "先にデータを読み込んでください"

data = raw_data.copy()

# --- 列名リネーム ---
rename_dict = {
    "id": "ID",
    "delta_week": "week",
    "fim": "FIM",
    "yo": "age",
    "male": "sex",
    "did_fall": "fall_any"
}
# 存在する列のみリネーム
rename_dict_valid = {k: v for k, v in rename_dict.items() if k in data.columns}
data = data.rename(columns=rename_dict_valid)

# --- 必須列の確認 ---
required_cols = ["ID", "week", "FIM"]
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f"必須列が見つかりません: {missing}. 現在の列: {list(data.columns)}")

# --- 型変換 ---
data["ID"] = data["ID"].astype(str)
data["week"] = pd.to_numeric(data["week"], errors="coerce")
data["FIM"] = pd.to_numeric(data["FIM"], errors="coerce")

# fall_any があれば数値化
if "fall_any" in data.columns:
    data["fall_any"] = pd.to_numeric(data["fall_any"], errors="coerce")

# --- 欠損除外 ---
data = data.dropna(subset=["ID", "week", "FIM"]).copy()

# --- (ID, week) 重複除外 ---
before = len(data)
data = data.drop_duplicates(subset=["ID", "week"], keep="first")
after = len(data)
print(f"(ID, week) 重複除外: {before} -> {after} 行 (削除: {before - after})")

# --- 3回以上観測のある患者のみ ---
id_counts = data["ID"].value_counts()
ids_ge3 = id_counts[id_counts >= 3].index
data = data[data["ID"].isin(ids_ge3)].copy()
data = data.sort_values(["ID", "week"]).reset_index(drop=True)

print(f"患者数 (>=3 observations): {data['ID'].nunique()}")
print(f"総観測数: {len(data)}")
print(f"Week 範囲: {data['week'].min()} - {data['week'].max()}")
print(f"FIM 範囲: {data['FIM'].min()} - {data['FIM'].max()}")


## 3. 全員FIM軌跡の表示


In [ ]:
# ====================================================
# 全員FIM軌跡の可視化
# ====================================================
fig_all, ax_all = plt.subplots(figsize=(12, 6))

for pid, grp in data.groupby("ID"):
    grp_sorted = grp.sort_values("week")
    ax_all.plot(grp_sorted["week"].values, grp_sorted["FIM"].values,
                alpha=0.15, linewidth=0.8, color="steelblue")

# 全体平均
mean_traj = data.groupby("week")["FIM"].mean().reset_index().sort_values("week")
ax_all.plot(mean_traj["week"], mean_traj["FIM"], color="red", linewidth=3,
            label="Overall Mean", zorder=10)

ax_all.set_xlabel("Week since admission", fontsize=12)
ax_all.set_ylabel("FIM Score", fontsize=12)
ax_all.set_title(f"FIM Trajectories (All {data['ID'].nunique()} patients)", fontsize=14)
ax_all.legend(fontsize=11)
ax_all.grid(True, alpha=0.3)
fig_all.tight_layout()
plt.show()


## 4. インタラクティブダッシュボード

### フィルタ条件
- **0週FIMスコアなし除外**: 0週のFIMスコアがない患者を除外するかどうか
- **0週FIMスコア範囲**: 0週のFIMスコアの下限・上限
- **Week上限**: 分析に含めるweekの上限

### 出力
- 情報量基準（BIC）による最適クラスタ数
- Individual Trajectories (sampled) + Class Means
- クラスタ別 Table 1


In [ ]:
# ====================================================
# ヘルパー関数
# ====================================================

def extract_trajectory_features(df):
    """
    患者ごとに FIM ~ week の線形回帰を行い、切片・傾き・観測数を特徴量として返す。
    LCMM の簡易近似として使用（高速）。
    """
    features = []
    for pid, grp in df.groupby("ID"):
        grp_s = grp.sort_values("week")
        weeks = grp_s["week"].values.astype(float)
        fims = grp_s["FIM"].values.astype(float)

        if len(weeks) < 2:
            features.append({
                "ID": pid,
                "intercept": fims[0],
                "slope": 0.0,
                "n_obs": len(weeks),
                "mean_fim": fims.mean(),
                "baseline_fim": fims[0]
            })
            continue

        try:
            slope, intercept = np.polyfit(weeks, fims, 1)
        except Exception:
            slope, intercept = 0.0, fims.mean()

        features.append({
            "ID": pid,
            "intercept": intercept,
            "slope": slope,
            "n_obs": len(weeks),
            "mean_fim": fims.mean(),
            "baseline_fim": fims[0]
        })

    return pd.DataFrame(features)


def find_optimal_clusters(features_scaled, max_k=6, min_k=1):
    """
    BIC で最適クラスタ数を選択する。
    Returns: best_k, bic_values dict, all fitted models dict
    """
    bic_values = {}
    models = {}

    for k in range(min_k, max_k + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            n_init=10,
            max_iter=300,
            random_state=42
        )
        gmm.fit(features_scaled)
        bic_values[k] = gmm.bic(features_scaled)
        models[k] = gmm

    best_k = min(bic_values, key=bic_values.get)
    return best_k, bic_values, models


def make_table1(baseline_df, group_col="FIM_class", id_col="ID"):
    """
    TableOne を使ってクラスタ別 Table 1 を生成する。
    """
    d = baseline_df.copy()
    d[group_col] = d[group_col].astype(str)

    # 除外列
    exclude = [id_col, group_col, "week", "intercept", "slope", "n_obs",
               "mean_fim", "baseline_fim", "FIM_class_label"]

    # 分析に使う列
    cols = [c for c in d.columns if c not in exclude and d[c].notna().sum() > 0]

    if len(cols) == 0:
        return None, "分析可能な列がありません"

    # カテゴリ変数の推定
    categorical = []
    for c in cols:
        s = d[c]
        if pd.api.types.is_bool_dtype(s):
            categorical.append(c)
        elif pd.api.types.is_numeric_dtype(s):
            if s.dropna().nunique() <= 10:
                categorical.append(c)
        else:
            categorical.append(c)

    try:
        table1 = TableOne(
            d,
            columns=cols,
            categorical=categorical,
            groupby=group_col,
            pval=True,
            missing=True
        )
        return table1, None
    except Exception as e:
        return None, str(e)


print("Helper functions defined.")


In [ ]:
# ====================================================
# インタラクティブダッシュボード
# ====================================================

# --- 0週のFIM範囲を事前計算 ---
week0_data = data[data["week"] == 0]
week0_ids = set(week0_data["ID"].unique())
has_week0_pct = len(week0_ids) / data["ID"].nunique() * 100

if len(week0_data) > 0:
    fim_min_w0 = int(week0_data["FIM"].min())
    fim_max_w0 = int(week0_data["FIM"].max())
else:
    fim_min_w0 = int(data["FIM"].min())
    fim_max_w0 = int(data["FIM"].max())

week_max_data = int(data["week"].max())

print(f"0週FIMあり患者: {len(week0_ids)}/{data['ID'].nunique()} ({has_week0_pct:.1f}%)")
print(f"0週FIM範囲: {fim_min_w0} - {fim_max_w0}")
print(f"最大week: {week_max_data}")

# ====================================================
# ウィジェット定義
# ====================================================
style = {'description_width': '200px'}
layout = widgets.Layout(width='500px')

w_exclude_no_week0 = widgets.Checkbox(
    value=False,
    description='0週FIMなし症例を除外',
    style=style,
    layout=layout
)

w_fim_range = widgets.IntRangeSlider(
    value=[fim_min_w0, fim_max_w0],
    min=fim_min_w0,
    max=fim_max_w0,
    step=1,
    description='0週FIMスコア範囲:',
    style=style,
    layout=layout,
    continuous_update=False
)

w_week_max = widgets.IntSlider(
    value=min(26, week_max_data),
    min=1,
    max=week_max_data,
    step=1,
    description='Week上限:',
    style=style,
    layout=layout,
    continuous_update=False
)

w_max_k = widgets.IntSlider(
    value=6,
    min=2,
    max=10,
    step=1,
    description='最大クラスタ数:',
    style=style,
    layout=layout,
    continuous_update=False
)

w_n_sample = widgets.IntSlider(
    value=30,
    min=5,
    max=100,
    step=5,
    description='表示サンプル数/クラス:',
    style=style,
    layout=layout,
    continuous_update=False
)

run_btn = widgets.Button(
    description='分析実行',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)

status_out = widgets.HTML(value='')

# 出力エリア
out_bic = widgets.Output()
out_traj = widgets.Output()
out_table1 = widgets.Output()

# ====================================================
# メイン分析関数
# ====================================================
def run_analysis(b=None):
    t_start = time.time()
    status_out.value = '<b style="color:blue">分析中...</b>'

    # 出力クリア
    out_bic.clear_output(wait=True)
    out_traj.clear_output(wait=True)
    out_table1.clear_output(wait=True)

    # --- フィルタ適用 ---
    df = data.copy()

    # Week上限フィルタ
    df = df[df["week"] <= w_week_max.value]

    # 0週FIMなし除外
    if w_exclude_no_week0.value:
        ids_with_w0 = set(df[df["week"] == 0]["ID"].unique())
        df = df[df["ID"].isin(ids_with_w0)]

    # 0週FIMスコア範囲フィルタ
    fim_lo, fim_hi = w_fim_range.value
    w0 = df[df["week"] == 0]
    if len(w0) > 0:
        ids_in_range = set(
            w0[(w0["FIM"] >= fim_lo) & (w0["FIM"] <= fim_hi)]["ID"].unique()
        )
        if w_exclude_no_week0.value:
            df = df[df["ID"].isin(ids_in_range)]
        else:
            ids_no_w0 = set(df["ID"].unique()) - set(w0["ID"].unique())
            df = df[df["ID"].isin(ids_in_range | ids_no_w0)]

    # 再度3点以上チェック
    id_counts = df["ID"].value_counts()
    ids_ge3 = id_counts[id_counts >= 3].index
    df = df[df["ID"].isin(ids_ge3)].copy()

    n_patients = df["ID"].nunique()
    n_obs = len(df)

    if n_patients < 3:
        status_out.value = f'<b style="color:red">フィルタ後の患者数が少なすぎます ({n_patients}人)。条件を緩和してください。</b>'
        return

    # --- 特徴量抽出 ---
    t_feat = time.time()
    feat_df = extract_trajectory_features(df)
    t_feat_done = time.time()

    # スケーリング
    scaler = StandardScaler()
    X = scaler.fit_transform(feat_df[["intercept", "slope"]].values)

    # --- 最適クラスタ数 ---
    t_clust = time.time()
    max_k = min(w_max_k.value, n_patients - 1)
    max_k = max(max_k, 2)
    best_k, bic_values, gmm_models = find_optimal_clusters(X, max_k=max_k)
    t_clust_done = time.time()

    # クラスラベル付与
    best_model = gmm_models[best_k]
    feat_df["cluster"] = best_model.predict(X)

    # クラスを平均FIMの昇順でソート（直感的に）
    cluster_means = feat_df.groupby("cluster")["mean_fim"].mean().sort_values()
    cluster_map = {old: new for new, old in enumerate(cluster_means.index)}
    feat_df["FIM_class"] = feat_df["cluster"].map(cluster_map)

    # データにクラスラベルを付与
    df = df.merge(feat_df[["ID", "FIM_class"]], on="ID", how="left")

    # --- BIC プロット ---
    with out_bic:
        clear_output(wait=True)
        fig_bic, ax_bic = plt.subplots(figsize=(8, 4))
        ks = sorted(bic_values.keys())
        bics = [bic_values[k] for k in ks]

        ax_bic.plot(ks, bics, 'o-', color='green', markersize=10, linewidth=2)
        ax_bic.axvline(x=best_k, color='red', linestyle='--', alpha=0.7,
                       label=f'Best: {best_k} classes (BIC={bic_values[best_k]:.1f})')
        ax_bic.scatter([best_k], [bic_values[best_k]], color='red', s=200, marker='*', zorder=10)
        ax_bic.set_xlabel("Number of Classes", fontsize=12)
        ax_bic.set_ylabel("BIC", fontsize=12)
        ax_bic.set_title("BIC by Number of Classes (lower = better)", fontsize=13)
        ax_bic.set_xticks(ks)
        ax_bic.legend(fontsize=11)
        ax_bic.grid(True, alpha=0.3)
        fig_bic.tight_layout()
        plt.show()

        # クラス分布表示
        class_counts = feat_df["FIM_class"].value_counts().sort_index()
        print(f"\n最適クラスタ数: {best_k} (BIC = {bic_values[best_k]:.1f})")
        print(f"患者数: {n_patients}, 観測数: {n_obs}")
        print(f"\nクラス分布:")
        for cls in sorted(class_counts.index):
            n = class_counts[cls]
            pct = n / n_patients * 100
            print(f"  Class {cls}: {n} patients ({pct:.1f}%)")

    # --- Trajectory プロット ---
    with out_traj:
        clear_output(wait=True)

        fig_traj, ax_traj = plt.subplots(figsize=(12, 7))

        # カラーマップ
        n_classes = best_k
        cmap = plt.cm.get_cmap('tab10', n_classes)
        colors = [cmap(i) for i in range(n_classes)]

        # クラス平均
        mean_traj = (
            df.dropna(subset=["FIM_class"])
            .groupby(["FIM_class", "week"], as_index=False)["FIM"]
            .mean()
            .sort_values(["FIM_class", "week"])
        )

        # サンプル個人軌跡
        np.random.seed(42)
        n_sample = w_n_sample.value

        for cls in sorted(df["FIM_class"].dropna().unique()):
            cls_int = int(cls)
            color = colors[cls_int % len(colors)]

            # サンプリング
            cls_ids = df[df["FIM_class"] == cls]["ID"].unique()
            sample_size = min(n_sample, len(cls_ids))
            sampled = np.random.choice(cls_ids, size=sample_size, replace=False)

            # 個人軌跡（薄く）
            for pid in sampled:
                pdata = df[df["ID"] == pid].sort_values("week")
                ax_traj.plot(pdata["week"].values, pdata["FIM"].values,
                            alpha=0.12, linewidth=0.8, color=color)

            # クラス平均（太く）
            cls_mean = mean_traj[mean_traj["FIM_class"] == cls]
            ax_traj.plot(cls_mean["week"], cls_mean["FIM"],
                        marker='o', linewidth=4, color=color,
                        label=f'Class {cls_int} (n={len(cls_ids)})',
                        zorder=10, markersize=6)

        ax_traj.set_xlabel("Week since admission", fontsize=12)
        ax_traj.set_ylabel("FIM Score", fontsize=12)
        ax_traj.set_title(
            f"Individual Trajectories (sampled, {n_sample}/class) + Class Means\n"
            f"({best_k} classes, {n_patients} patients, week <= {w_week_max.value})",
            fontsize=13
        )
        ax_traj.legend(fontsize=11, loc='best')
        ax_traj.grid(True, alpha=0.3)
        fig_traj.tight_layout()
        plt.show()

    # --- Table 1 ---
    with out_table1:
        clear_output(wait=True)

        # ベースラインデータ作成（week==0優先、なければ最小week）
        baseline_parts = []

        # week==0 がある患者
        d0 = df[df["week"] == 0].copy()
        if len(d0) > 0:
            d0_base = d0.sort_values(["ID", "week"]).drop_duplicates(subset=["ID"], keep="first")
            baseline_parts.append(d0_base)
            ids_with_w0 = set(d0_base["ID"].unique())
        else:
            ids_with_w0 = set()

        # week==0 がない患者 → 最小weekを使用
        df_no_w0 = df[~df["ID"].isin(ids_with_w0)]
        if len(df_no_w0) > 0:
            d_min = df_no_w0.sort_values(["ID", "week"]).drop_duplicates(subset=["ID"], keep="first")
            baseline_parts.append(d_min)

        if baseline_parts:
            baseline = pd.concat(baseline_parts, axis=0).reset_index(drop=True)
        else:
            baseline = df.sort_values(["ID", "week"]).drop_duplicates(subset=["ID"], keep="first").reset_index(drop=True)

        # FIM_class ラベル
        baseline["FIM_class_label"] = baseline["FIM_class"].apply(
            lambda x: f"Class {int(x)}" if pd.notna(x) else "NA"
        )
        baseline = baseline.dropna(subset=["FIM_class"])

        print(f"=== Table 1 ({best_k} classes, {len(baseline)} patients) ===\n")

        table1_obj, err = make_table1(baseline, group_col="FIM_class_label", id_col="ID")

        if table1_obj is not None:
            print(table1_obj.tabulate(tablefmt="grid"))
        else:
            print(f"Table 1 生成エラー: {err}")

    t_end = time.time()
    status_out.value = (
        f'<b style="color:green">完了!</b> '
        f'({n_patients}人, {n_obs}obs, {best_k}クラス) '
        f'| 特徴量抽出: {(t_feat_done-t_feat)*1000:.0f}ms '
        f'| クラスタリング: {(t_clust_done-t_clust)*1000:.0f}ms '
        f'| 合計: {(t_end-t_start):.2f}s'
    )

run_btn.on_click(run_analysis)

# ====================================================
# レイアウト
# ====================================================
controls = widgets.VBox([
    widgets.HTML(value='<h3>フィルタ設定</h3>'),
    w_exclude_no_week0,
    w_fim_range,
    w_week_max,
    widgets.HTML(value='<h3>分析設定</h3>'),
    w_max_k,
    w_n_sample,
    run_btn,
    status_out
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', width='550px'))

outputs = widgets.VBox([
    widgets.HTML(value='<h3>BIC によるクラスタ数選択</h3>'),
    out_bic,
    widgets.HTML(value='<h3>Individual Trajectories + Class Means</h3>'),
    out_traj,
    widgets.HTML(value='<h3>Table 1 (クラスタ別ベースライン特性)</h3>'),
    out_table1
], layout=widgets.Layout(padding='10px', width='100%'))

dashboard = widgets.VBox([
    widgets.HTML(value='<h2>FIM軌跡 インタラクティブダッシュボード</h2>'),
    widgets.HBox([controls, outputs])
])

display(dashboard)
print("\n分析実行ボタンを押してダッシュボードを更新してください。")


## 5. リアクティブモード（オプション）

下のセルを実行すると、ウィジェットの値が変更されるたびに自動で分析が再実行されます。  
（初回は上の「分析実行」ボタンで手動実行してください）


In [ ]:
# ====================================================
# リアクティブモード: ウィジェット変更時に自動再実行
# ====================================================
# debounce用のタイマー
_last_change_time = [0.0]
_debounce_ms = 500  # 500ms debounce

def _on_widget_change(change):
    import threading
    _last_change_time[0] = time.time()
    t = _last_change_time[0]

    def _delayed_run():
        time.sleep(_debounce_ms / 1000.0)
        if _last_change_time[0] == t:
            run_analysis()

    thread = threading.Thread(target=_delayed_run, daemon=True)
    thread.start()

# ウィジェットにオブザーバーを登録
for w in [w_exclude_no_week0, w_fim_range, w_week_max, w_max_k, w_n_sample]:
    w.observe(_on_widget_change, names='value')

print("リアクティブモード有効: ウィジェット変更で自動再分析されます (debounce: 500ms)")
